# 大模型后训练：从 SFT 到 DPO 与 GRPO

> **本章定位**：使用已有基座模型或 `40_pre_training.ipynb` 的模型制品，沿统一资产链建立 Supervised Fine-Tuning（SFT，监督微调）、Direct Preference Optimization（DPO，直接偏好优化）与 Group Relative Policy Optimization（GRPO，组相对策略优化）的数据和损失契约。

> **章节边界**：PEFT 参数化策略、推理轨迹训练、训练系统优化与模型压缩分别见 `42_peft.ipynb`、`A30_reasoning_model.ipynb`、`A40_training_optimization.ipynb` 与 `A60_model_compression.ipynb`；本章不重复这些工程机制。

> **总览**：内容以同一条版本化资产链组织三类目标：SFT 学习任务格式与目标回答，DPO 根据离线 chosen/rejected 偏好对调整策略，GRPO 通过同一 Prompt 的成组 Rollout 与可验证奖励优化序列级行为；随后将原理契约映射到 Hugging Face TRL。

```mermaid
flowchart LR
    B["预训练基座 + 固定 Tokenizer"] --> S["SFT<br/>指令—回答监督"]
    S --> C["SFT Checkpoint / Adapter"]
    C --> D["DPO<br/>离线 chosen / rejected"]
    D --> A["对齐后 Checkpoint / Adapter"]
    A --> G["GRPO<br/>成组 Rollout + Reward"]
    G --> R["候选发布产物"]
    P["PEFT 参数化策略"] -.-> S
    P -.-> D
    P -.-> G
    R --> E["质量、安全、格式与性能验收"]
```


## 1．学习契约

| 项目 | 内容 |
|---|---|
| 路线 | 模型训练与适配：监督微调与对齐 |
| 本章定位 | 沿同一资产链建立 SFT、DPO、GRPO 的数据与目标函数边界。 |
| 先修知识 | 掌握 `31` 的 Causal LM、log-probability、梯度更新与训练/验证隔离；训练循环、检查点与恢复知识见 `40`。基座可来自已有模型或自行预训练的制品。 |
| 预计时间 | 2～3 小时 |
| 运行资源 | 小型原理模型可在 CPU 或 Colab 运行；正式训练通常需要 GPU。 |
| 输入 | 指令回答、偏好对、成组 Rollout 与奖励。 |
| 交付物 | SFT、DPO、GRPO 策略产物与统一评测契约。 |

### 1.1．学习目标

完成本章后，读者能够区分 SFT、DPO 与 GRPO 的数据结构、参考策略和优化目标，实现并验证三类损失契约，将原理对象映射到 TRL 配置，并建立可恢复、可评估的后训练资产链。


In [ ]:
# 基础环境。后续主路径只依赖这些包。
import copy
import math
import platform
import random
from contextlib import nullcontext
from dataclasses import dataclass
from typing import Iterable, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import Dataset, DatasetDict

# 固定数据划分、初始化与采样；质量比较需使用预先登记的多个种子。
SEED = 42  # 仅支持实验重放，不保证跨硬件、Kernel 或分布式拓扑逐位一致。
DEVICE = torch.accelerator.current_accelerator(check_available=True) or torch.device("cpu")

random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
elif DEVICE.type == "mps":
    torch.mps.manual_seed(SEED)

print({
    "python": platform.python_version(),
    "torch": torch.__version__,
    "device": str(DEVICE),
})


### 1.2．环境与依赖

本章依赖以 `../requirements.txt` 为准，其中 TRL 固定为已验收版本。安装或更新依赖后需要重启 Kernel，以确保运行时加载同一组版本。


In [ ]:
# 按需安装后训练框架依赖，安装后重启 Kernel。
# %pip install -r ../requirements.txt


## 2．直觉与输入输出契约

### 2.1．后训练资产链

后训练由四类相互约束的资产组成：基座与 Tokenizer、数据与模板、目标函数与参考策略、评估与版本化产物。各阶段沿同一资产链传递，而非仅按 Trainer 调用顺序连接。

```mermaid
flowchart TD
    B["版本化基座"] --> S1["SFT 数据：messages / prompt-completion"]
    S1 --> S2["回答区域 Label Mask"]
    S2 --> S3["Token-level Cross Entropy"]
    S3 --> C1["SFT Checkpoint"]
    C1 --> D1["DPO 数据：prompt / chosen / rejected"]
    D1 --> D2["Policy 与 Reference Log-prob 差"]
    D2 --> C2["DPO Checkpoint"]
    C2 --> G1["GRPO Prompt + 成组 Rollout"]
    G1 --> G2["Reward + 组内标准化优势"]
    G2 --> C3["GRPO Checkpoint"]
    C3 --> E["离线评测 + 安全评测 + 发布闸门"]
```


### 2.2．任务、数据与最小 Tokenizer

本节构造一个完全离线且数据转换可逐项检查的监督微调任务。

输入为少量“用户问题 → 助手回答”样本，输出为整数 Token 序列及 `PAD/BOS/EOS/UNK` 四个特殊 Token。字符级 Tokenizer 虽然编码效率有限，但不依赖外部网络，能够完整展示编码与解码过程。固定样例应满足 `decode(encode(text)) == text`，且训练集与验证集不存在重叠；生产环境则使用与基座模型严格匹配的 Tokenizer。


In [ ]:
# 小型指令数据；生产数据应做去重、质量过滤、隐私与许可证审查。
records = [
    {"prompt": "把 2 加 3 等于多少？", "response": "2 加 3 等于 5。"},
    {"prompt": "把 7 加 4 等于多少？", "response": "7 加 4 等于 11。"},
    {"prompt": "什么是梯度累积？", "response": "梯度累积把多个微批次的梯度合并后再更新参数。"},
    {"prompt": "什么是混合精度？", "response": "混合精度用较低精度计算，并在关键位置保留足够精度。"},
    {"prompt": "LoRA 的核心思想是什么？", "response": "LoRA 用两个低秩矩阵学习权重增量，并冻结原始权重。"},
    {"prompt": "KV Cache 有什么作用？", "response": "KV Cache 复用历史 Token 的键和值，避免重复计算。"},
    {"prompt": "什么是吞吐量？", "response": "吞吐量表示单位时间内系统处理的 Token 或请求数量。"},
    {"prompt": "什么是首 Token 延迟？", "response": "首 Token 延迟是请求到达后生成第一个 Token 所需的时间。"},
]

# 前 6 条训练、后 2 条验证是固定成员契约；正式数据应按用户、会话或业务实体分组隔离。
# 用 DatasetDict 固化拆分名称，后续变换始终保留 train/validation 边界。
dataset_dict = DatasetDict({
    "train": Dataset.from_list(records[:6]),
    "validation": Dataset.from_list(records[6:]),
})
train_records = dataset_dict["train"]
valid_records = dataset_dict["validation"]


In [ ]:
# 最小字符级 Tokenizer。
class MyCharTokenizer:
    # 按固定顺序注册特殊 Token，并建立字符词表。
    """构建仅覆盖给定文本的最小字符级 Tokenizer，并维护字符与特殊 Token 的双向映射。"""
    def __init__(self, texts: Iterable[str]):
        """从语料收集字符词表，并初始化特殊 Token 的编号。"""
        specials = ["<pad>", "<bos>", "<eos>", "<unk>"]
        chars = sorted(set("".join(texts)))
        self.id_to_token = specials + chars
        self.token_to_id = {token: idx for idx, token in enumerate(self.id_to_token)}
        self.pad_token_id = self.token_to_id["<pad>"]
        self.bos_token_id = self.token_to_id["<bos>"]
        self.eos_token_id = self.token_to_id["<eos>"]
        self.unk_token_id = self.token_to_id["<unk>"]

    @property
    def vocab_size(self) -> int:
        """返回包含特殊 Token 在内的词表大小。"""
        return len(self.id_to_token)

    def encode(self, text: str) -> list[int]:
        """将字符串逐字符映射为 Token ID，未登录字符使用未知 Token。"""
        return [self.token_to_id.get(char, self.unk_token_id) for char in text]

    def decode(self, ids: Iterable[int], skip_special_tokens: bool = True) -> str:
        """将 Token ID 还原为字符串，并可过滤特殊 Token。"""
        specials = {"<pad>", "<bos>", "<eos>", "<unk>"}
        tokens = [self.id_to_token[int(idx)] for idx in ids]
        if skip_special_tokens:
            tokens = [token for token in tokens if token not in specials]
        return "".join(tokens)


def my_format_prompt(prompt: str) -> str:
    """把用户输入格式化为固定的监督微调提示模板。"""
    return f"用户：{prompt}\n助手："


all_texts = [
    text
    for record in records
    for text in (my_format_prompt(record["prompt"]), record["response"])
]
tokenizer = MyCharTokenizer(all_texts)

probe = "什么是吞吐量？"
print("vocab_size =", tokenizer.vocab_size)
print(tokenizer.encode(probe), "->", tokenizer.decode(tokenizer.encode(probe)))


<!-- theory-math-contract:v1 -->
### 2.3．核心机制的语言与数学表达

后训练方法改变监督信号，但都需要明确样本单位和归约口径。SFT 只对回答区域计算 Token 损失；DPO 比较策略相对参考模型对优选与拒选回答的对数概率差：

$$
\mathcal L_{\mathrm{SFT}}=-\frac{1}{\sum m_t}\sum_t m_t\log\pi_\theta(y_t\mid x,y_{<t})
$$

$$
\mathcal L_{\mathrm{DPO}}=-\log\sigma\!\left(\beta\left[\log\frac{\pi_\theta(y^+\mid x)}{\pi_{\mathrm{ref}}(y^+\mid x)}-\log\frac{\pi_\theta(y^-\mid x)}{\pi_{\mathrm{ref}}(y^-\mid x)}\right]\right)
$$

其中，$m_t$ 是回答区 Label Mask，$y^+,y^-$ 是偏好对，$\beta$ 控制偏好间隔尺度。`completion_mask`、序列 Log-probability 与 TRL 的 `SFTTrainer`/`DPOTrainer` 分别对应这些对象。公式不消除数据偏差；错误偏好、长度偏差和奖励投机仍需切片评测。

以上数学表示用于明确变量、形状与约束；实际结论仍需由本章的数值、形状、梯度、性能或失败案例证据验证。

## 3．最小原理实现

### 3.1．回答区域的 Label Mask

**Label Mask（标签掩码）**：把不应计入损失的位置设为 `-100`；PyTorch 交叉熵会忽略这些位置。

格式化后的提示与回答转换为 `input_ids`、`attention_mask` 和 `labels`。若提示部分参与损失，模型会额外学习复述会话模板；监督微调通常只监督目标回答。因此，提示与 Padding 位置的标签设为 `-100`，回答正文与 `EOS` 保留原 Token ID。

<!-- diagram:sft-data-contract -->
SFT 只让回答区域参与损失，Prompt 仍作为条件输入模型：

![架构图：SFT 会话模板、Tokenizer、标签屏蔽与训练批次契约](assets/figures/41_post_training/sft-data-contract.svg)

[TikZ 源文件](assets/figures/41_post_training/sft-data-contract.tex)


In [ ]:
# 单样本编码与动态 Padding Collator。
IGNORE_INDEX = -100  # System、User 与 Padding 的损失屏蔽值，必须与交叉熵 ignore_index 一致。
SFT_MAX_LENGTH = 128  # 覆盖本章短对话；降低会增加回答截断，提高会增加 Padding 与 Attention 成本。


def my_encode_sft_record(record: dict, max_length: int = SFT_MAX_LENGTH) -> dict[str, list[int]]:
    """将一条提示—回答记录编码为定长上限内的 input_ids 与 labels，并屏蔽提示区域的监督。"""
    prompt_ids = tokenizer.encode(my_format_prompt(record["prompt"]))
    answer_ids = tokenizer.encode(record["response"])
    input_ids = [tokenizer.bos_token_id] + prompt_ids + answer_ids + [tokenizer.eos_token_id]
    labels = [IGNORE_INDEX] * (1 + len(prompt_ids)) + answer_ids + [tokenizer.eos_token_id]

    input_ids = input_ids[:max_length]
    labels = labels[:max_length]
    return {"input_ids": input_ids, "labels": labels}


class MySFTCollator:
    # 保存批次 Padding 协议。
    """把变长 SFT 样本动态补齐为批张量，并同步生成注意力掩码与监督标签。"""
    def __init__(self, pad_token_id: int, pad_to_multiple_of: Optional[int] = None):
        """记录 Padding Token 及可选的长度对齐倍数。"""
        self.pad_token_id = pad_token_id
        self.pad_to_multiple_of = pad_to_multiple_of

    def __call__(self, examples: list[dict[str, list[int]]]) -> dict[str, torch.Tensor]:
        """按批内最大长度补齐样本，返回形状为 [batch, sequence] 的输入、掩码和标签张量。"""
        max_len = max(len(example["input_ids"]) for example in examples)
        if self.pad_to_multiple_of:
            m = self.pad_to_multiple_of
            max_len = math.ceil(max_len / m) * m

        input_ids, attention_mask, labels = [], [], []
        # 按当前批次最大长度组装矩形张量。
        for example in examples:
            pad_len = max_len - len(example["input_ids"])
            input_ids.append(example["input_ids"] + [self.pad_token_id] * pad_len)
            attention_mask.append([1] * len(example["input_ids"]) + [0] * pad_len)
            labels.append(example["labels"] + [IGNORE_INDEX] * pad_len)

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


tokenized = dataset_dict.map(
    my_encode_sft_record,
    remove_columns=dataset_dict["train"].column_names,
)
encoded_train = tokenized["train"]
encoded_valid = tokenized["validation"]
collator = MySFTCollator(tokenizer.pad_token_id)
batch = collator([encoded_train[idx] for idx in range(min(3, len(encoded_train)))])

print({name: tuple(value.shape) for name, value in batch.items()})


In [ ]:
# 展示一条样本的提示掩码、回答监督与 Padding 区域。
sample = collator([encoded_train[idx] for idx in range(min(2, len(encoded_train)))])
row = 0
tokens = [tokenizer.id_to_token[idx] for idx in sample["input_ids"][row].tolist()]
roles = [
    "_" if mask == 0 else ("P" if label == IGNORE_INDEX else "A")
    for mask, label in zip(sample["attention_mask"][row].tolist(), sample["labels"][row].tolist())
]
print("".join(tokens))
print("".join(roles))


#### 3.1.1．Prompt、回答与 Padding 的监督边界

学习问题是：同一条会话序列中哪些位置为模型提供 SFT 梯度。下图直接读取上一单元的 `attention_mask` 与 `labels`，将位置分为 Padding、只作为条件的 Prompt、参与交叉熵的回答三类。验收条件是所有 Padding 标签均为 `-100`，且 Prompt 与回答区域都非空。


In [ ]:
# 将真实 Collator 输出转换为三态监督图，不重新实现 Label Mask。
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

row_attention = sample["attention_mask"][row].detach().cpu()
row_labels = sample["labels"][row].detach().cpu()
padding_positions = row_attention.eq(0)
prompt_positions = row_attention.eq(1) & row_labels.eq(IGNORE_INDEX)
answer_positions = row_labels.ne(IGNORE_INDEX)
if bool((padding_positions & row_labels.ne(IGNORE_INDEX)).any()):
    raise RuntimeError("Padding 位置仍包含有效监督标签")
if not bool(prompt_positions.any()) or not bool(answer_positions.any()):
    raise RuntimeError("SFT 样本缺少 Prompt 条件区或回答监督区")
role_values = torch.zeros_like(row_labels)
role_values[prompt_positions] = 1
role_values[answer_positions] = 2

fig, ax = plt.subplots(figsize=(14, 2.6))
ax.imshow(role_values[None, :], cmap=ListedColormap(["#BDBDBD", "#56B4E9", "#E69F00"]), vmin=0, vmax=2, aspect="auto")
ax.set_yticks([0], ["监督角色"])
ax.set_xticks(range(len(tokens)), tokens, rotation=90, fontsize=8)
ax.set_xlabel("Token 位置")
ax.set_title("SFT 中 Prompt 仅提供条件，回答 Token 参与 Loss")
ax.legend(
    handles=[Patch(color="#BDBDBD", label="Padding"), Patch(color="#56B4E9", label="Prompt：label=-100"), Patch(color="#E69F00", label="Answer：参与 Loss")],
    loc="upper center", bbox_to_anchor=(0.5, -0.45), ncol=3,
)
plt.show()
print({"prompt_tokens": int(prompt_positions.sum()), "answer_tokens": int(answer_positions.sum()), "padding_tokens": int(padding_positions.sum())})


Prompt 区域虽然不直接进入交叉熵，仍通过因果 Attention 影响回答位置的 Hidden State。图中橙色位置表示监督信号的直接来源，不表示只有这些 Token 参与前向计算。Label Mask 正确也不能证明会话模板、截断策略或答案质量已经正确，仍需分别验收。


### 3.2．Dynamic Padding 与 Packing

- **Dynamic Padding（动态填充）**：每个批次仅填充到本批最长序列，实现清晰且边界明确。
- **Packing（序列打包）**：把多个短样本装入同一固定长度序列，减少 Padding 浪费，但必须隔离样本边界，避免后一个样本关注前一个样本或错误地跨样本计算损失。

生产建议：先以动态填充建立正确基线；只有分析出 Padding 浪费明显后再启用库级 Packing，并验证 attention mask、position ids、EOS 边界与指标一致性。


In [ ]:
# 量化 Padding 浪费；这是是否值得 Packing 的第一条证据。
def my_padding_report(encoded: list[dict], batch_size: int) -> dict[str, float]:
    """统计分批动态 Padding 前后的 Token 数，并返回 Padding 浪费比例。"""
    real, padded = 0, 0
    # 每个批次按最长样本补齐，累计真实 token 与补齐后 token 数。
    for start in range(0, len(encoded), batch_size):
        stop = min(start + batch_size, len(encoded))
        part = [encoded[idx] for idx in range(start, stop)]
        max_len = max(len(item["input_ids"]) for item in part)
        real += sum(len(item["input_ids"]) for item in part)
        padded += max_len * len(part)
    return {
        "real_tokens": real,
        "padded_tokens": padded,
        "padding_ratio": 1.0 - real / padded,
    }


print(my_padding_report(encoded_train, batch_size=3))


### 3.3．最小 Causal LM 与 SFT 基线

**Causal Language Model（因果语言模型）**只能关注当前位置及之前的 Token。

模型接收形状为 `[batch, sequence]` 的 Token ID 与 Padding Mask，输出 `[batch, sequence, vocab]` 的 logits，并计算右移一位后的交叉熵损失。显式实现 Q/K/V、因果 Mask、残差、LayerNorm 和逐 Token 损失，可为后续优化提供可比较基线。正确性证据包括张量形状一致、未来 Token 的注意力概率为零、一次反向传播后的梯度为有限值，以及训练损失下降。


In [ ]:
# 训练基线使用最小因果自注意力；推理优化专题再单独实现 KV Cache。
class MyCausalSelfAttention(nn.Module):
    # 注册 QKV 投影与输出投影。
    """实现带因果掩码和可选 Padding 掩码的多头自注意力。"""
    def __init__(self, d_model: int, n_heads: int, dropout: float):
        """创建合并的 QKV 投影、输出投影，并记录头数、头维度与 Dropout。"""
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.dropout = dropout

    # 在分头空间计算带因果与 Padding Mask 的注意力。
    def forward(self, x: torch.Tensor, attention_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """将 [batch, sequence, hidden] 隐状态变换为同形状的因果自注意力输出。"""
        batch_size, sequence_length, d_model = x.shape
        qkv = self.qkv(x).view(batch_size, sequence_length, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)
        q, k, v = (tensor.transpose(1, 2) for tensor in (q, k, v))

        scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_dim)
        causal_mask = torch.triu(
            torch.ones(sequence_length, sequence_length, dtype=torch.bool, device=x.device),
            diagonal=1,
        )
        scores = scores.masked_fill(causal_mask[None, None], torch.finfo(scores.dtype).min)
        if attention_mask is not None:
            scores = scores.masked_fill(
                (attention_mask[:, None, None, :] == 0),
                torch.finfo(scores.dtype).min,
            )

        probs = F.softmax(scores, dim=-1)
        probs = F.dropout(probs, p=self.dropout, training=self.training)
        out = probs @ v
        out = out.transpose(1, 2).contiguous().view(batch_size, sequence_length, d_model)
        return self.proj(out)


class MyTransformerBlock(nn.Module):
    # 组合 Pre-Norm 注意力与 MLP 子层。
    """组合 Pre-LN 因果自注意力、前馈网络与两条残差连接。"""
    def __init__(self, d_model: int, n_heads: int, mlp_ratio: int, dropout: float):
        """初始化归一化层、注意力层和扩张后回投影的 MLP。"""
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MyCausalSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, mlp_ratio * d_model),
            nn.GELU(),
            nn.Linear(mlp_ratio * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor, attention_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """依次执行注意力与 MLP 残差更新，返回与输入同形状的隐藏状态。"""
        x = x + self.attn(self.ln1(x), attention_mask=attention_mask)
        return x + self.mlp(self.ln2(x))


In [ ]:
# 只包含微调路径的最小 Causal LM。
from torch.utils.checkpoint import checkpoint


@dataclass
class MyCausalLMOutput:
    # 执行前向计算，得到后续损失或解码需要的模型输出。
    """封装语言模型的逐 Token logits 与可选训练损失。"""
    logits: torch.Tensor
    loss: Optional[torch.Tensor] = None


# 64 维、4 头、2 层、256 个位置与四倍 MLP 构成结构完整的小型模型；扩大配置会增加显存与训练时间。
class MyTinyCausalLM(nn.Module):
    # 注册 Embedding、Transformer Block、归一化与共享输出层。
    """实现用于训练机制验证的小型 Decoder-only Causal LM。"""
    def __init__(
        self,
        vocab_size: int,
        d_model: int = 64,
        n_heads: int = 4,
        n_layers: int = 2,
        max_length: int = 256,
        dropout: float = 0.0,
    ):
        """初始化 Token/位置嵌入、Transformer Block、输出归一化和共享权重语言模型头。"""
        super().__init__()
        self.max_length = max_length
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_length, d_model)
        self.blocks = nn.ModuleList([
            MyTransformerBlock(d_model, n_heads, mlp_ratio=4, dropout=dropout)
            for _ in range(n_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding.weight
        self.gradient_checkpointing = False

    # 构造位置表示，并计算移位后的因果语言模型损失。
    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        labels: Optional[torch.Tensor] = None,
    ) -> MyCausalLMOutput:
        """计算 [batch, sequence, vocab] logits，并在提供 labels 时返回移位后的因果语言模型损失。"""
        _, sequence_length = input_ids.shape
        positions = torch.arange(sequence_length, device=input_ids.device)
        x = self.token_embedding(input_ids) + self.position_embedding(positions)[None]

        for block in self.blocks:
            if self.gradient_checkpointing and self.training:
                x = checkpoint(
                    lambda hidden, current_block=block: current_block(
                        hidden, attention_mask=attention_mask
                    ),
                    x,
                    use_reentrant=False,
                )
            else:
                x = block(x, attention_mask=attention_mask)

        logits = self.lm_head(self.final_norm(x))
        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=IGNORE_INDEX,
            )
        return MyCausalLMOutput(logits=logits, loss=loss)


model = MyTinyCausalLM(tokenizer.vocab_size).to(DEVICE)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable parameters: {trainable:,}")


In [ ]:
# 观察一次前向、反向和梯度清零的数据流。
probe_batch = {name: tensor[:2].to(DEVICE) for name, tensor in batch.items()}
output = model(**probe_batch)
output.loss.backward()

model.zero_grad(set_to_none=True)
print("loss =", float(output.loss.detach()))


### 3.4．最小训练循环

训练循环只保留必要动作：取批次、前向、反向、梯度裁剪、参数更新。本例每卡 Batch 为 2、无梯度累积且只有一个数据并行进程，所以有效 Batch 为 2；6 条训练记录、1 个 Epoch 对应 3 次更新。序列长短不一时，同样的样本 Batch 可能包含完全不同的监督 Token 数，生产比较应同时记录每步有效 Token。`max_norm=1.0` 是梯度尖峰保护的起点，不是固定最佳值：若多数步骤都被裁剪，先检查学习率、数据异常与数值精度，再决定是否调大；若仍出现非有限梯度或尖峰，则应调小并定位原因。生产训练还要补充 Warmup、Scheduler、断点续训、分布式采样、实验追踪、容错、数据版本与随机状态保存。


In [ ]:
# 小数据训练循环，保持更新路径完全可观察。
def my_iter_minibatches(encoded: list[dict], batch_size: int, shuffle: bool = True):
    """按可选随机顺序切分编码样本，并逐批产出已动态补齐的张量。"""
    indices = list(range(len(encoded)))
    if shuffle:
        random.shuffle(indices)
    for start in range(0, len(indices), batch_size):
        yield collator([encoded[idx] for idx in indices[start : start + batch_size]])


def my_move_batch(batch: dict[str, torch.Tensor], device: torch.device):
    """把批字典中的全部张量迁移到目标设备并保持键不变。"""
    return {name: tensor.to(device) for name, tensor in batch.items()}


SFT_LEARNING_RATE = 3e-3  # 随机初始化微型模型的有限步学习率；预训练基座通常需要更低取值。
SFT_WEIGHT_DECAY = 0.01  # 当前正则起点，应在固定有效 Token 预算下复核。
SFT_EPOCHS = 1  # 6 条样本配合 Batch 2 仅产生 3 次更新，用于验证链路而非质量结论。
SFT_BATCH_SIZE = 2  # 单卡且无累积时有效 Batch 为 2；长度差异大时还需比较每步有效 Token。
SFT_MAX_GRAD_NORM = 1.0  # 异常梯度保护上限；持续裁剪时应检查学习率与损失尺度。
baseline_model = MyTinyCausalLM(tokenizer.vocab_size).to(DEVICE)
optimizer = torch.optim.AdamW(
    baseline_model.parameters(), lr=SFT_LEARNING_RATE, weight_decay=SFT_WEIGHT_DECAY
)
epochs = SFT_EPOCHS
loss_history = []

baseline_model.train()
# 每个批次执行一次完整参数更新。
for epoch in range(epochs):
    for cpu_batch in my_iter_minibatches(encoded_train, batch_size=SFT_BATCH_SIZE):
        train_batch = my_move_batch(cpu_batch, DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss = baseline_model(**train_batch).loss
        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(baseline_model.parameters(), max_norm=SFT_MAX_GRAD_NORM)
        optimizer.step()
        loss_history.append(float(loss.detach()))

print({"first_loss": loss_history[0], "last_loss": loss_history[-1], "steps": len(loss_history)})


## 4．证据验证

### 4.1．Loss、Perplexity 与生成结果

**Perplexity（困惑度）**定义为平均 Token 负对数似然的指数。它适合比较同一 Tokenizer、同一数据处理下的模型；不同 Tokenizer 的困惑度不可直接横比。

评估输入为未参与参数更新的验证集，输出包括按有效回答 Token 加权的损失、困惑度和生成文本。直接平均各批次损失会使短批次获得不当权重，因此实现累计 NLL 总和与有效 Token 数。评估在 `model.eval()` 和 `torch.inference_mode()` 下执行，标签为 `-100` 的位置不计入分母。`exp(min(mean_nll, 20))` 仅用于避免展示时发生指数溢出；报告仍应保留未截断 NLL，并标记困惑度是否触及显示上限。当前验证集只有 2 条记录，只能验证评估链路，不能支持质量比较。


In [ ]:
# Token 加权评估。
@torch.inference_mode()
def my_evaluate_nll(model: nn.Module, encoded: list[dict], batch_size: int = 2) -> dict[str, float]:
    """按有效监督 Token 加权汇总验证集 NLL，并返回困惑度与 Token 数。"""
    model.eval()
    total_nll = 0.0
    total_tokens = 0
    # 逐批处理数据，控制单次计算规模并累计阶段结果。
    for cpu_batch in my_iter_minibatches(encoded, batch_size=batch_size, shuffle=False):
        eval_batch = my_move_batch(cpu_batch, DEVICE)
        output = model(**eval_batch)
        valid_tokens = (eval_batch["labels"][:, 1:] != IGNORE_INDEX).sum().item()
        total_nll += float(output.loss) * valid_tokens
        total_tokens += valid_tokens
    mean_nll = total_nll / max(total_tokens, 1)
    return {
        "nll": mean_nll,
        "perplexity": math.exp(min(mean_nll, 20)),
        "supervised_tokens": total_tokens,
    }


print(my_evaluate_nll(baseline_model, encoded_valid))


### 4.2．后训练方法的验证坐标

三种方法改变的训练信号不同，PEFT 则改变“更新哪些参数”，两者不能混为同一维度。

| 方法 | 输入数据 | 训练信号 | 是否需要 Reference / Reward | 主要产物 |
|---|---|---|---|---|
| SFT | System/User/Assistant 或 Prompt/Completion | 目标回答 Token 的交叉熵 | 不需要 | 指令模型或 Adapter |
| DPO | Prompt、Chosen、Rejected | 策略相对参考模型的偏好间隔 | 需要冻结 Reference；不单独训练 Reward Model | 偏好对齐模型或 Adapter |
| GRPO | Prompt、同 Prompt 的多条 Rollout、可验证或学习型奖励 | 组内相对优势、裁剪策略目标与 KL 约束 | 需要 Reward；通常保留 Reference 约束 | 在线策略 Checkpoint 或 Adapter |
| PEFT | 与 SFT/DPO/GRPO 相同 | 不改变上面的任务目标 | 取决于所配合的方法 | 基座 + Adapter |

```mermaid
flowchart TD
    Q{"可用监督信号"}
    Q -->|"只有高质量目标回答"| S["SFT"]
    Q -->|"有离线偏好对"| D["DPO"]
    Q -->|"可生成 Rollout 且有可靠 Reward"| G["GRPO"]
    S --> N{"是否需要继续对齐"}
    N -->|"离线偏好充分"| D
    N -->|"在线探索有价值"| G
    P{"训练状态与交付预算"} -->|"轻量任务资产"| E["PEFT"]
    E -.-> S
    E -.-> D
    E -.-> G
```

DPO 不能替代 SFT 建立基础任务能力；GRPO 也不能用不稳定、可被投机的 Reward 弥补数据和任务定义缺陷。每次阶段切换都必须先固定上游产物哈希、数据版本与独立评测基线。


### 4.3．DPO 偏好目标验证

DPO 数据契约是同一个 `prompt` 下的 `chosen` 与 `rejected`。Policy 和冻结 Reference 分别计算两条回答的条件 Log-prob；优化目标提高 Policy 相对 Reference 对 Chosen 的偏好间隔：

$$
L_{\mathrm{DPO}}
=
-\log \sigma\left(
\beta
\left[
\log\frac{\pi_\theta(y_w|x)}{\pi_{\mathrm{ref}}(y_w|x)}
-
\log\frac{\pi_\theta(y_l|x)}{\pi_{\mathrm{ref}}(y_l|x)}
\right]
\right)
$$

<!-- diagram:dpo-preference-objective -->

![架构图：DPO 中 Policy 与冻结 Reference 对偏好对的评分与损失聚合](assets/figures/41_post_training/dpo-preference-objective.svg)

[TikZ 源文件](assets/figures/41_post_training/dpo-preference-objective.tex)

偏好对必须来自相同 Prompt 和兼容的采样条件；模板、截断或长度差异不能成为标签泄漏。本章除 DPO Loss 外，还观测 Chosen/Rejected Reward、Margin、KL、回答长度和独立任务质量。`beta=0.1` 是起始尺度，而非概率阈值；其职责也不同于后文 GRPO 中作为 KL 系数的同名参数。DPO Beta 变化后，需在相同数据、有效 Batch 与 Token 预算下重新比较偏好准确率、策略—Reference KL、长度分布和任务回归。


In [ ]:
# 从零实现 DPO Loss：输入为按回答 Token 求和后的四组序列 Log-prob。
# beta=0.1 是相对冻结 Reference 的偏好间隔尺度；应联合 Reward Margin、KL 与任务回归调优。
def my_dpo_loss(
    policy_chosen_logps,
    policy_rejected_logps,
    reference_chosen_logps,
    reference_rejected_logps,
    beta=0.1,
):
    """根据策略与参考模型的 chosen/rejected 对数概率计算 DPO 损失、偏好准确率和奖励间隔。"""
    policy_margin = policy_chosen_logps - policy_rejected_logps
    reference_margin = reference_chosen_logps - reference_rejected_logps
    logits = beta * (policy_margin - reference_margin)
    losses = -F.logsigmoid(logits)
    chosen_rewards = beta * (
        policy_chosen_logps - reference_chosen_logps
    ).detach()
    rejected_rewards = beta * (
        policy_rejected_logps - reference_rejected_logps
    ).detach()
    return losses.mean(), {
        "chosen_reward": chosen_rewards.mean(),
        "rejected_reward": rejected_rewards.mean(),
        "reward_margin": (chosen_rewards - rejected_rewards).mean(),
    }


# 两个真实张量维度表示 batch 内两组偏好对，只验证公式的数据流。
policy_chosen = torch.tensor([-2.1, -1.8], device=DEVICE)
policy_rejected = torch.tensor([-2.8, -2.4], device=DEVICE)
reference_chosen = torch.tensor([-2.3, -2.0], device=DEVICE)
reference_rejected = torch.tensor([-2.6, -2.2], device=DEVICE)
dpo_loss, dpo_metrics = my_dpo_loss(
    policy_chosen,
    policy_rejected,
    reference_chosen,
    reference_rejected,
)
print({"loss": float(dpo_loss), **{k: float(v) for k, v in dpo_metrics.items()}})


#### 4.3.1．相对偏好间隔、Beta 与 DPO Loss

学习问题是：Policy 相对 Reference 更偏向 Chosen 时，DPO Loss 如何响应，以及 `beta` 如何改变响应尺度。下图逐点调用上一单元的 `my_dpo_loss`，只改变相对偏好间隔。验收条件是每条曲线随间隔增大而单调下降，并在间隔为 0 时等于 $\log 2$。


In [ ]:
# 通过原理 Loss 函数扫描相对偏好间隔，比较不同 beta 的尺度效应。
import matplotlib.pyplot as plt

relative_margins = torch.linspace(-6.0, 6.0, 121, device=DEVICE)
zero_logp = torch.zeros(1, device=DEVICE)
beta_values = (0.05, 0.1, 0.5)
dpo_curves = {}
for beta_value in beta_values:
    dpo_curves[beta_value] = [
        float(
            my_dpo_loss(
                margin.reshape(1), zero_logp, zero_logp, zero_logp, beta=beta_value
            )[0].detach().cpu()
        )
        for margin in relative_margins
    ]
    if any(left < right for left, right in zip(dpo_curves[beta_value], dpo_curves[beta_value][1:])):
        raise RuntimeError(f"beta={beta_value} 的 DPO Loss 未随偏好间隔单调下降")
zero_index = int(relative_margins.abs().argmin().detach().cpu().item())
if abs(dpo_curves[0.1][zero_index] - math.log(2.0)) > 1e-6:
    raise RuntimeError("相对偏好间隔为 0 时 DPO Loss 不等于 log(2)")

fig, ax = plt.subplots(figsize=(8.5, 4.2))
colors = ("#0072B2", "#009E73", "#D55E00")
for beta_value, color in zip(beta_values, colors):
    ax.plot(relative_margins.detach().cpu(), dpo_curves[beta_value], color=color, label=f"beta={beta_value}")
ax.axvline(0.0, color="black", linewidth=1.0, linestyle=":")
ax.axhline(math.log(2.0), color="#666666", linewidth=1.0, linestyle="--", label="margin=0 时 log(2)")
ax.set(title="DPO 对相对偏好间隔的响应", xlabel="Policy Margin − Reference Margin", ylabel="DPO Loss")
ax.grid(alpha=0.25)
ax.legend()
plt.show()
print({"loss_at_zero_margin": dpo_curves[0.1][zero_index], "expected": math.log(2.0)})


较大的 `beta` 会让同一相对间隔产生更陡的 Loss 变化，但并不表示偏好数据更可靠或模型质量更高。曲线只解释目标函数的局部尺度；生产选择仍需联合观察策略—Reference KL、长度分布、独立任务质量、安全回归与梯度稳定性。


### 4.4．GRPO 组内优势验证

GRPO 对每个 Prompt 采样一组回答，用规则、验证器、Reward Model 或环境结果打分，再在组内标准化得到相对优势。它省去独立 Value Model，但没有消除 Reward 设计、在线生成、Reference KL 和策略稳定性的成本。

对同一 Prompt 的奖励 $r_1,\ldots,r_G$，组内优势为：

$$
A_i = \frac{r_i-\operatorname{mean}(r_{1:G})}
{\operatorname{std}(r_{1:G})+\epsilon}
$$

```mermaid
flowchart LR
    P["Prompt"] --> G["Policy 生成 G 条 Rollout"]
    G --> R1["规则 / Verifier Reward"]
    G --> R2["Reward Model"]
    G --> R3["安全与格式 Reward"]
    R1 --> S["加权 Reward"]
    R2 --> S
    R3 --> S
    S --> A["组内标准化 Advantage"]
    A --> O["Clipped Policy Objective + KL"]
    O --> U["Policy Update"]
```

Reward 必须版本化并接受对抗测试。格式奖励不能压过任务正确性；可验证答案要防止答案泄漏；学习型 Reward Model 要监控分布漂移。必须同时保存原始 Rollout、各 Reward 分量、总分、优势、KL 与生成配置。本章每组两条 Rollout 与 $[0,1]$ 区间内的固定奖励只用于验证最小链路：同组奖励若全部相同，标准化优势就会全部为零。生产应监控零方差组占比，并用候选数、采样温度、奖励分辨率和生成成本的联合曲线决定配置。


In [ ]:
# 从零实现组内优势与裁剪策略目标，张量形状为 [batch, group, completion_tokens]。
def my_group_relative_advantages(rewards, epsilon=1e-6):
    """在每个提示的候选组内标准化奖励，返回零均值的相对优势。"""
    group_mean = rewards.mean(dim=1, keepdim=True)
    group_std = rewards.std(dim=1, keepdim=True, unbiased=False)
    return (rewards - group_mean) / group_std.clamp_min(epsilon)


# clip=0.2 与 KL beta=0.04 是策略更新保护的起点；需结合奖励尺度、KL 与有效组比例调优。
def my_grpo_loss(
    new_logps,
    old_logps,
    reference_logps,
    advantages,
    clip_epsilon=0.2,
    kl_beta=0.04,
):
    """用组内相对优势加权新旧策略概率比，并结合 KL 正则计算 GRPO 目标。"""
    ratio = torch.exp(new_logps - old_logps)
    clipped_ratio = torch.clamp(
        ratio,
        1.0 - clip_epsilon,
        1.0 + clip_epsilon,
    )
    token_advantages = advantages.unsqueeze(-1)
    policy_objective = torch.minimum(
        ratio * token_advantages,
        clipped_ratio * token_advantages,
    )

    # 非负 KL 估计用于约束策略偏离冻结 Reference。
    reference_gap = reference_logps - new_logps
    kl = torch.exp(reference_gap) - reference_gap - 1.0
    loss = -(policy_objective - kl_beta * kl).mean()
    return loss, {
        "mean_advantage": advantages.mean().detach(),
        "mean_kl": kl.mean().detach(),
    }


# 每组 2 个 Rollout 是组内相对优势的最小规模；增大会降低估计方差但线性增加生成成本。
# 固定形状：rewards.shape = [2, 2]。
rewards = torch.tensor([[1.0, 0.2], [0.4, 0.9]], device=DEVICE)
advantages = my_group_relative_advantages(rewards)
# 固定形状：old_logps.shape = [2, 2, 3]。
old_logps = torch.full((2, 2, 3), -1.5, device=DEVICE)
new_logps = old_logps + torch.tensor(
    [[[0.05], [-0.03]], [[-0.02], [0.04]]],
    device=DEVICE,
)
reference_logps = old_logps - 0.01
grpo_loss, grpo_metrics = my_grpo_loss(
    new_logps,
    old_logps,
    reference_logps,
    advantages,
)
print({
    "advantages": advantages.tolist(),
    "loss": float(grpo_loss),
    **{k: float(v) for k, v in grpo_metrics.items()},
})


#### 4.4.1．组内 Reward 与相对 Advantage

学习问题是：GRPO 如何把不同绝对尺度的组内 Reward 转换为可比较的相对 Advantage。下图直接使用上一单元的 `rewards` 与 `advantages`。验收条件是每个非零方差组的 Advantage 均值接近 0；同组中高于均值的 Rollout 为正，低于均值的 Rollout 为负。


In [ ]:
# 并排呈现同一批真实 Rollout 的 Reward 与组内标准化 Advantage。
import matplotlib.pyplot as plt

reward_values = rewards.detach().float().cpu()
advantage_values = advantages.detach().float().cpu()
group_mean_error = float(advantage_values.mean(dim=1).abs().max())
if group_mean_error > 1e-5:
    raise RuntimeError(f"组内 Advantage 均值未归零：{group_mean_error:.2e}")

group_count, rollout_count = reward_values.shape
x_positions = torch.arange(rollout_count).numpy()
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.0), sharex=True)
colors = ("#0072B2", "#D55E00", "#009E73", "#CC79A7")
for group_index in range(group_count):
    offset = (group_index - (group_count - 1) / 2) * 0.16
    axes[0].bar(x_positions + offset, reward_values[group_index], width=0.16, color=colors[group_index], label=f"Prompt group {group_index}")
    axes[1].bar(x_positions + offset, advantage_values[group_index], width=0.16, color=colors[group_index], label=f"Prompt group {group_index}")
axes[0].set(title="原始 Reward", xlabel="Rollout 索引", ylabel="Reward")
axes[1].set(title="组内标准化 Advantage", xlabel="Rollout 索引", ylabel="Advantage")
axes[1].axhline(0.0, color="black", linewidth=1.0, linestyle=":")
for ax in axes:
    ax.set_xticks(x_positions)
    ax.grid(axis="y", alpha=0.25)
axes[0].legend()
plt.tight_layout()
plt.show()
print({"reward_shape": tuple(reward_values.shape), "max_group_mean_advantage_error": group_mean_error})


Advantage 表示同一 Prompt 组内的相对位置，不能跨组直接解释为绝对质量差。同组 Reward 全部相同时方差为 0，本实现会得到全 0 Advantage，该组不提供排序更新信号；生产训练必须单独监控零方差组占比、奖励投机、Reward 尺度、KL 与生成成本。


## 5．迁移到生产库

### 5.1．映射到 Hugging Face TRL

TRL 贯穿 SFT、DPO 与 GRPO，PEFT 通过 `peft_config` 横切三类 Trainer。数据列、Tokenizer Chat Template、Reference、Reward 函数和保存策略必须显式固定。

| 训练阶段 | TRL 接口 | 最小数据契约 | 关键配置 |
|---|---|---|---|
| SFT | `SFTTrainer` / `SFTConfig` | `messages` 或 Prompt/Completion | Assistant-only Loss、Packing、最大长度 |
| DPO | `DPOTrainer` / `DPOConfig` | `prompt`、`chosen`、`rejected` | `beta`、Reference、截断策略、Loss 类型 |
| GRPO | `GRPOTrainer` / `GRPOConfig` | `prompt` 与 Reward 所需附加列 | `num_generations`、Reward 权重、KL、生成与裁剪配置 |
| 横切 PEFT | Trainer 的 `peft_config` | 与对应阶段相同 | Adapter 类型、目标模块、Rank、量化与保存契约 |

<!-- diagram:trl-training-boundary -->

![架构图：TRL Trainer 的版本化输入契约、训练接口与输出制品](assets/figures/41_post_training/trl-training-boundary.svg)

[TikZ 源文件](assets/figures/41_post_training/trl-training-boundary.tex)

库迁移按模型 ID 加载 `HuggingFaceTB/SmolLM2-135M-Instruct`。Trainer 单元下载模型并执行极短链路验证，用于检查接口、数据、生成、反向传播和保存路径；该结果不构成模型质量或发布结论。


In [ ]:
# 准备 TRL 的真实 Dataset 契约并按模型 ID 加载上游当前文件。
from datasets import Dataset
from transformers import AutoTokenizer

# 135M 指令模型用于资源受限的真实接口验证；更换模型后需复核模板、显存与质量基线。
MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
)
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token

dpo_dataset = Dataset.from_list([
    {
        "prompt": "用户：请用一句话解释为什么生产模型需要记录制品哈希。\\n助手：",
        "chosen": "制品哈希能验证权重和配置内容，并支持可靠回滚。",
        "rejected": "因为 main 分支一般不会变化。",
    },
    {
        "prompt": "用户：线上服务发现新模型质量下降，第一步应该做什么？\\n助手：",
        "chosen": "停止继续放量，保留证据并回滚到已验收的不可变版本。",
        "rejected": "立即删除全部日志并继续观察。",
    },
])

grpo_dataset = Dataset.from_list([
    {
        "prompt": [{"role": "user", "content": "计算 17 + 25，只输出整数。"}],
        "solution": "42",
    },
    {
        "prompt": [{"role": "user", "content": "计算 9 × 7，只输出整数。"}],
        "solution": "63",
    },
])
print({
    "dpo_rows": len(dpo_dataset),
    "grpo_rows": len(grpo_dataset),
})


In [ ]:
# 使用 TRL DPOTrainer，并通过 PEFT 只训练 LoRA Adapter。
from peft import LoraConfig, TaskType
from trl import DPOConfig, DPOTrainer

post_training_peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=4,
    lora_alpha=8,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
)

# 两次更新仅验证 TRL 链路；max_steps 会覆盖 epoch，正式训练应以有效 Token 与验证停止策略定预算。
dpo_args = DPOConfig(
    output_dir="artifacts/post_training/dpo",
    model_init_kwargs={
        "dtype": "auto",
    },
    max_steps=2,  # 两次参数更新只验证数据、反向传播与保存链路。
    per_device_train_batch_size=1,  # 单卡有效 Batch 为 1；生产值需结合显存与有效 Token 调整。
    gradient_accumulation_steps=1,  # 不累积梯度；改变后需联动复核学习率。
    learning_rate=1e-6,  # 真实预训练模型的保守起点，应依据验证曲线重新选择。
    beta=0.1,  # DPO 偏好尺度，与 GRPO 的 KL beta 职责不同。
    max_length=128,  # 覆盖本章短偏好对；调整后需复核截断率与显存。
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    disable_tqdm=False,
    seed=SEED,
)
dpo_trainer = DPOTrainer(
    model=MODEL_ID,
    args=dpo_args,
    train_dataset=dpo_dataset,
    processing_class=tokenizer,
    peft_config=post_training_peft_config,
)
dpo_result = dpo_trainer.train()
dpo_trainer.save_model("artifacts/post_training/dpo/final")
print(dpo_result.metrics)


In [ ]:
# 使用可验证答案 Reward 运行 TRL GRPOTrainer。
from trl import GRPOConfig, GRPOTrainer


# 二值 0/1 奖励便于核对精确答案，但稀疏反馈与两候选组只支持链路验证。
def my_exact_answer_reward(completions, solution, **kwargs):
    """对完成文本做规范化后与标准答案精确匹配，逐条返回二值奖励。"""
    contents = [
        completion[0]["content"].strip()
        for completion in completions
    ]
    return [
        1.0 if content == expected.strip() else 0.0
        for content, expected in zip(contents, solution, strict=True)
    ]


# 两次更新仅验证生成、奖励与反向传播；正式预算应按有效 Rollout 和冻结验证集确定。
grpo_args = GRPOConfig(
    output_dir="artifacts/post_training/grpo",
    model_init_kwargs={
        "dtype": "auto",
    },
    max_steps=2,  # 两次参数更新只验证完整链路。
    per_device_train_batch_size=2,  # 每设备 Prompt 数；还会被候选数放大为 Rollout 成本。
    gradient_accumulation_steps=1,  # 不累积梯度；改变后需联动复核学习率。
    learning_rate=1e-6,  # 真实预训练模型的保守起点，应依据质量与 KL 曲线复核。
    num_generations=2,  # 组内比较下限；增加会近似线性提高生成成本。
    max_completion_length=32,  # 足以容纳整数答案；提高会增加延迟与奖励噪声。
    temperature=0.8,  # 用于形成候选差异；过低会同质化，过高会增加无效答案。
    epsilon=0.2,  # 策略比率裁剪起点，应结合有效组比例与质量曲线调优。
    beta=0.04,  # GRPO 的 Reference KL 系数；增大会强化策略约束。
    remove_unused_columns=False,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    disable_tqdm=False,
    seed=SEED,
)
grpo_trainer = GRPOTrainer(
    model=MODEL_ID,
    args=grpo_args,
    reward_funcs=my_exact_answer_reward,
    train_dataset=grpo_dataset,
    processing_class=tokenizer,
    peft_config=post_training_peft_config,
)
grpo_result = grpo_trainer.train()
grpo_trainer.save_model("artifacts/post_training/grpo/final")
print(grpo_result.metrics)


## 6．生产边界

### 6.1．后训练生产验收

| 维度 | SFT | DPO | GRPO |
|---|---|---|---|
| 数据 | 来源、许可证、去重、模板、回答 Mask | Prompt 一致性、偏好来源、标注一致性、长度偏差 | Prompt 分布、Rollout 采样、Reward 输入与环境版本 |
| 目标 | Validation NLL、任务与格式质量 | Chosen/Rejected Reward、Margin、KL、任务回归 | Reward 分量、优势方差、KL、熵、Clip Fraction |
| 行为 | 指令遵循、安全、幻觉 | 偏好提升与过度拒答、风格偏移 | Reward Hacking、模式坍塌、探索不足与安全绕过 |
| 系统 | 有效 Token/s、峰值显存、恢复 | Policy/Reference 显存、Log-prob 吞吐 | 生成吞吐、Rollout 队列、Reward 延迟与失败率 |
| 产物 | 完整模型或 Adapter | Policy/Adapter + Reference ID 与哈希 | Policy/Adapter + Reward/环境版本/生成配置 |
| 发布 | 与基座同集回归 | 与 SFT 基线同集回归 | 与 SFT、DPO 基线同时比较并保留回滚 |

训练损失下降或 Reward 上升均不能单独作为发布依据。每个阶段必须使用独立 Holdout、对抗集、安全集和真实负载评测；任何 Reward、模板、Tokenizer、Reference 或环境变更都需要触发重新验收。表中的 2 条 DPO 偏好对和 2 条 GRPO Prompt 仅用于验证 Schema 与训练链路。正式评测应按任务、长度、语言和风险切片预先确定样本量，报告分子、分母、置信区间和相对冻结基线的变化；高严重度失效使用独立硬门禁，不能被平均分抵消，也不能在反复变更 Seed 后仅报告最优运行。


### 6.2．参考资料（官方文档）

- [TRL Quickstart](https://huggingface.co/docs/trl/quickstart)
- [TRL SFTTrainer](https://huggingface.co/docs/trl/sft_trainer)
- [TRL DPOTrainer](https://huggingface.co/docs/trl/dpo_trainer)
- [TRL GRPOTrainer](https://huggingface.co/docs/trl/grpo_trainer)
- [TRL Reward Functions](https://huggingface.co/docs/trl/rewards)
- [Hugging Face Datasets Process](https://huggingface.co/docs/datasets/process)
- [Transformers Causal Language Modeling](https://huggingface.co/docs/transformers/tasks/language_modeling)
- [PEFT Methods Overview](https://huggingface.co/docs/peft/main/methods/overview)

生产使用前，以 `../requirements.txt` 锁定版本对应的官方文档和回归结果为准。


### 6.3．本章小结

后训练的主线是：SFT 用目标回答建立任务能力和格式，DPO 用离线偏好对调整策略相对 Reference 的回答排序，GRPO 用成组 Rollout 和可靠 Reward 优化序列级行为。三者共享基座、Tokenizer、模板、评测和版本化资产链，但训练信号不能混淆。

PEFT 是可横切 SFT、DPO、GRPO 的参数化策略；TRL 是承载这些目标的标准训练库。最终交付必须绑定上游模型 ID、制品哈希、数据与 Reward 契约、训练状态、评测证据和回滚路径。
